# Enterprise Audio Anti-Spoofing Forensics: Light-CNN + LFCC
## End-to-End Deepfake Detection Pipeline on ASVspoof 2019 LA Benchmark

### Theoretical Foundation
Human speech production conforms to the linear source-filter model:
$$s(t) = e(t) * h(t)$$
where $e(t)$ represents the glottal excitation pulse and $h(t)$ denotes the vocal tract acoustic filter.

Synthetic speech engines (Text-to-Speech and Voice Conversion) introduce characteristic high-frequency spectral artifacts and phase inconsistencies during neural vocoding (e.g. WaveNet, WORLD). While standard Mel-scale filterbanks compress high-frequency resolution according to psychoacoustic perception, **Linear Frequency Cepstral Coefficients (LFCC)** employ uniformly spaced triangular filterbanks across the full Nyquist range ($0 - 8000\text{ Hz}$). This retains high-frequency artifact traces critical for synthetic voice detection.

The **Light-CNN** architecture employs **Max-Feature-Map (MFM)** activation functions:
$$\text{MFM}(x) = \max(x_{2k-1}, x_{2k})$$
MFM operates as an intrinsic feature selector and noise filter by suppressing low-magnitude stochastic noise while preserving deterministic vocoder artifact patterns with minimal parameter overhead.

### 1. Environment Setup and Hardware Verification
Initializes random seeds, verifies available GPU acceleration, and installs necessary signal processing libraries.

In [ ]:
import subprocess
subprocess.run(["pip", "install", "soundfile", "librosa", "-q"])

import os, glob, time, json, math, random
import numpy as np, pandas as pd
import soundfile as sf, librosa, scipy.fftpack as fft_
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score, accuracy_score, precision_recall_fscore_support

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda"
print(f"Compute Device: {device}")
if use_amp:
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


### 2. Dataset Discovery and Path Resolution
Locates the ASVspoof 2019 Logical Access dataset directory, supporting direct mounts, linked datasets, and nested paths.

In [ ]:
def locate_la_root():
    candidates = [
        "/kaggle/input/asvpoof-2019-dataset/LA",
        "/kaggle/input/datasets/awsaf49/asvpoof-2019-dataset/LA/LA",
        "/kaggle/input/datasets/awsaf49/asvpoof-2019-dataset/LA",
        "/kaggle/input/asvpoof2019-dataset/LA",
        "/kaggle/input/asvpoof-2019/LA"
    ]
    for pattern in ["/kaggle/input/**/ASVspoof2019_LA_cm_protocols", "/kaggle/input/**/ASVspoof2019_LA_train"]:
        for match in glob.glob(pattern, recursive=True):
            candidates.insert(0, os.path.dirname(match))
    for c in candidates:
        if os.path.isdir(c) and os.path.isdir(os.path.join(c, "ASVspoof2019_LA_train")):
            return os.path.abspath(c)
    return "/kaggle/input"

la_root = locate_la_root()

def get_proto_file(root, part, suffix):
    direct = os.path.join(root, "ASVspoof2019_LA_cm_protocols", f"ASVspoof2019.LA.cm.{part}.{suffix}.txt")
    if os.path.isfile(direct):
        return direct
    matches = glob.glob(f"/kaggle/input/**/ASVspoof2019.LA.cm.{part}.{suffix}.txt", recursive=True)
    if matches:
        return matches[0]
    matches = glob.glob(f"/kaggle/input/**/*{part}*.txt", recursive=True)
    for m in matches:
        base = os.path.basename(m).lower()
        if "cm" in base and not base.startswith("._"):
            return m
    return direct

def get_flac_dir(root, part):
    direct = os.path.join(root, f"ASVspoof2019_LA_{part}", "flac")
    if os.path.isdir(direct):
        return direct
    matches = glob.glob(f"/kaggle/input/**/ASVspoof2019_LA_{part}/flac", recursive=True)
    if matches:
        return matches[0]
    matches = glob.glob(f"/kaggle/input/**/ASVspoof2019_LA_{part}", recursive=True)
    for m in matches:
        sub = os.path.join(m, "flac")
        if os.path.isdir(sub):
            return sub
        return m
    return direct

flac_dirs = {
    "train": get_flac_dir(la_root, "train"),
    "dev": get_flac_dir(la_root, "dev"),
    "eval": get_flac_dir(la_root, "eval")
}

proto_files = {
    "train": get_proto_file(la_root, "train", "trn"),
    "dev": get_proto_file(la_root, "dev", "trl"),
    "eval": get_proto_file(la_root, "eval", "trl")
}

print(f"Resolved LA Root: {la_root}")
for k in ["train", "dev", "eval"]:
    p_ok = os.path.isfile(proto_files[k])
    d_ok = os.path.isdir(flac_dirs[k])
    d_count = len(os.listdir(flac_dirs[k])) if d_ok else 0
    print(f"  [{k.upper()}] Protocol: {'OK' if p_ok else 'MISSING'} ({proto_files[k]})")
    print(f"          Audio:    {'OK' if d_ok else 'MISSING'} ({d_count:,} files in {flac_dirs[k]})")
assert os.path.isfile(proto_files["train"]), f"Missing training protocol: {proto_files['train']}"
assert os.path.isfile(proto_files["dev"]), f"Missing dev protocol: {proto_files['dev']}"


### 3. Protocol Parsing and Data Integrity Audit
Parses space-delimited ground truth protocol manifests into structured data frames.

In [ ]:
rows = []
for partition, path in proto_files.items():
    if not os.path.exists(path):
        continue
    flac_folder = flac_dirs[partition]
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            spk, aid, env, atk, key = parts[0], parts[1], parts[2], parts[3], parts[4]
            rows.append({
                "speaker_id": spk,
                "audio_id": aid,
                "environment_id": env,
                "attack_id": atk,
                "key": key,
                "is_spoof": 1 if key == "spoof" else 0,
                "partition": partition,
                "file_path": os.path.join(flac_folder, f"{aid}.flac")
            })

manifest = pd.DataFrame(rows)
assert len(manifest) > 0, "Protocol parsing produced 0 records. Check dataset paths."
print(f"Total Parsed Records: {len(manifest):,}")
summary = []
for part in ["train", "dev", "eval"]:
    sub = manifest[manifest["partition"] == part]
    if len(sub) == 0:
        continue
    bon = int((sub["key"] == "bonafide").sum())
    spf = int((sub["key"] == "spoof").sum())
    summary.append({
        "Partition": part,
        "Total": len(sub),
        "Bonafide": bon,
        "Spoof": spf,
        "Spoof:Bonafide Ratio": f"{spf / max(bon, 1):.2f}:1"
    })
print(pd.DataFrame(summary).to_string(index=False))


### 4. Acoustic Preprocessing and LFCC Feature Extraction Pipeline
Standardizes audio signals into deterministic 4.0-second (64,000 samples) pre-emphasized arrays and computes Linear Frequency Cepstral Coefficients ($60 \times 251$).

In [ ]:
def load_raw_audio(path, target_sr=16000):
    y, orig_sr = sf.read(path)
    if y.ndim > 1:
        y = y.mean(axis=1)
    y = y.astype(np.float32)
    if orig_sr != target_sr:
        y = librosa.resample(y, orig_sr=orig_sr, target_sr=target_sr)
    return y

def preprocess_audio(y, is_train=False, target_len=64000, alpha=0.97, top_db=40):
    y = np.concatenate([[y[0]], y[1:] - alpha * y[:-1]])
    intervals = librosa.effects.split(y=y, top_db=top_db)
    if len(intervals):
        trimmed = np.concatenate([y[s:e] for s, e in intervals])
        if len(trimmed) > 1000:
            y = trimmed
    n_samples = len(y)
    if n_samples >= target_len:
        start = np.random.randint(0, n_samples - target_len + 1) if is_train else (n_samples - target_len) // 2
        y = y[start:start + target_len]
    else:
        y = np.pad(y, (0, target_len - n_samples), mode="wrap")
    return y / (np.max(np.abs(y)) + 1e-7)

def extract_lfcc(y, sr=16000, n_fft=1024, hop_length=256, n_ceps=20, max_frames=251):
    stft = librosa.stft(y, n_fft=n_fft, hop_length=hop_length, center=True)
    power_spec = np.abs(stft) ** 2
    n_bins = power_spec.shape[0]
    filterbank = np.zeros((n_ceps, n_bins), dtype=np.float32)
    points = np.linspace(0, n_bins - 1, n_ceps + 2, dtype=int)
    for i in range(n_ceps):
        lo, mid, hi = points[i], points[i + 1], points[i + 2]
        if mid > lo:
            filterbank[i, lo:mid] = np.linspace(0, 1, mid - lo)
        if hi > mid:
            filterbank[i, mid:hi] = np.linspace(1, 0, hi - mid)
    linear_energies = np.maximum(filterbank @ power_spec, 1e-8)
    static = fft_.dct(np.log(linear_energies), type=2, axis=0, norm="ortho")[:n_ceps]
    delta1 = librosa.feature.delta(static, order=1)
    delta2 = librosa.feature.delta(static, order=2)
    features = np.vstack([static, delta1, delta2]).astype(np.float32)
    if features.shape[1] >= max_frames:
        features = features[:, :max_frames]
    else:
        features = np.pad(features, ((0, 0), (0, max_frames - features.shape[1])), mode="edge")
    return features

test_row = manifest[manifest["partition"] == "train"].iloc[0]
test_audio = preprocess_audio(load_raw_audio(test_row["file_path"]), is_train=False)
test_lfcc = extract_lfcc(test_audio)
print(f"Sanity Check Audio Shape: {test_audio.shape} (Expected: (64000,))")
print(f"Sanity Check LFCC Shape:  {test_lfcc.shape} (Expected: (60, 251))")
assert test_audio.shape == (64000,) and test_lfcc.shape == (60, 251)


### 5. PyTorch Dataset with SpecAugment and Balanced Sampling
Applies time and frequency masking to prevent overfitting on known synthesis algorithms.

In [ ]:
class LFCCDataset(Dataset):
    def __init__(self, df, is_train=False):
        self.df = df.reset_index(drop=True)
        self.is_train = is_train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            raw = load_raw_audio(row["file_path"])
            proc = preprocess_audio(raw, is_train=self.is_train)
            feat = extract_lfcc(proc)
        except Exception:
            feat = np.zeros((60, 251), dtype=np.float32)

        if self.is_train:
            if np.random.rand() < 0.5:
                w_t = np.random.randint(1, 31)
                t_0 = np.random.randint(0, max(1, 251 - w_t))
                feat[:, t_0:t_0 + w_t] = feat.mean()
            if np.random.rand() < 0.5:
                w_f = np.random.randint(1, 11)
                f_0 = np.random.randint(0, max(1, 60 - w_f))
                feat[f_0:f_0 + w_f, :] = feat.mean()

        tensor_x = torch.from_numpy(feat).unsqueeze(0)
        tensor_y = torch.tensor(int(row["is_spoof"]), dtype=torch.long)
        return tensor_x, tensor_y

train_df = manifest[manifest["partition"] == "train"].reset_index(drop=True)
dev_df = manifest[manifest["partition"] == "dev"].reset_index(drop=True)

train_dataset = LFCCDataset(train_df, is_train=True)
dev_dataset = LFCCDataset(dev_df, is_train=False)

train_targets = train_df["is_spoof"].values
class_counts = np.bincount(train_targets)
class_weights = 1.0 / class_counts
sample_weights = torch.FloatTensor(class_weights[train_targets])
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

batch_size = 128
train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=sampler, num_workers=2, pin_memory=True)
dev_loader = DataLoader(dev_dataset, batch_size=batch_size * 2, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train Batches per Epoch: {len(train_loader):,}")
print(f"Dev Batches per Epoch:   {len(dev_loader):,}")


### 6. Light-CNN Architecture with Max-Feature-Map (MFM)
Implements convolutional feature transformation with competitive Max-Feature-Map activations.

In [ ]:
class MFM(nn.Module):
    def forward(self, x):
        c1, c2 = torch.split(x, x.size(1) // 2, dim=1)
        return torch.max(c1, c2)

class LightCNN(nn.Module):
    def __init__(self, in_channels=1, num_classes=2, dropout=0.3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=5, stride=1, padding=2),
            MFM(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(32, 64, kernel_size=1),
            MFM(),
            nn.BatchNorm2d(32),
            nn.Conv2d(32, 96, kernel_size=3, stride=1, padding=1),
            MFM(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.BatchNorm2d(48),
            nn.Conv2d(48, 96, kernel_size=1),
            MFM(),
            nn.BatchNorm2d(48),
            nn.Conv2d(48, 128, kernel_size=3, stride=1, padding=1),
            MFM(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(64, 128, kernel_size=1),
            MFM(),
            nn.BatchNorm2d(64),
            nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1),
            MFM(),
            nn.BatchNorm2d(32),
            nn.Conv2d(32, 64, kernel_size=1),
            MFM(),
            nn.BatchNorm2d(32),
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            MFM(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.AdaptiveAvgPool2d(1)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

model = LightCNN().to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total Trainable Parameters: {total_params:,}")

with torch.no_grad():
    dummy_input = torch.zeros(2, 1, 60, 251).to(device)
    dummy_out = model(dummy_input)
    print(f"Forward Pass Test Shape:    {dummy_out.shape} (Expected: (2, 2))")


### 7. Loss Formulation, Optimization and Evaluation Metrics
Implements Focal Loss with label smoothing to stabilize gradient backpropagation on imbalanced classes.

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0, label_smoothing=0.05):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.label_smoothing = label_smoothing

    def forward(self, logits, targets):
        ce_loss = F.cross_entropy(logits, targets, reduction="none", label_smoothing=self.label_smoothing)
        p_t = torch.exp(-ce_loss)
        alpha_factor = torch.where(targets == 1, self.alpha, 1.0 - self.alpha)
        focal_loss = alpha_factor * ((1.0 - p_t) ** self.gamma) * ce_loss
        return focal_loss.mean()

def calculate_eer(y_true, y_score):
    fpr, tpr, thresholds = roc_curve(y_true, y_score, pos_label=1)
    fnr = 1 - tpr
    idx = np.nanargmin(np.abs(fpr - fnr))
    eer_val = float((fpr[idx] + fnr[idx]) / 2)
    optimal_thresh = float(thresholds[idx])
    return eer_val, optimal_thresh

def evaluate_network(model, loader, device, use_amp):
    model.eval()
    scores_list, targets_list = [], []
    with torch.no_grad():
        for x_batch, y_batch in loader:
            x_batch = x_batch.to(device)
            with torch.amp.autocast(device_type=device.type, enabled=use_amp):
                probs = torch.softmax(model(x_batch), dim=1)[:, 1]
            scores_list.append(probs.cpu().numpy())
            targets_list.append(y_batch.numpy())
    scores = np.concatenate(scores_list)
    targets = np.concatenate(targets_list)
    eer, thresh = calculate_eer(targets, scores)
    auc = float(roc_auc_score(targets, scores))
    return eer, thresh, auc, scores, targets


### 8. End-to-End Model Training Loop
Trains Light-CNN over 30 epochs with Cosine Annealing learning rate schedule and automatic mixed precision.

In [ ]:
epochs = 30
lr_init = 1e-3
lr_min = 1e-6
weight_decay = 1e-4

criterion = FocalLoss(alpha=0.75, gamma=2.0, label_smoothing=0.05)
optimizer = torch.optim.AdamW(model.parameters(), lr=lr_init, weight_decay=weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=lr_min)
scaler = torch.amp.GradScaler(device=device.type, enabled=use_amp)

save_dir = "/kaggle/working"
os.makedirs(save_dir, exist_ok=True)
best_checkpoint_path = os.path.join(save_dir, "light_cnn_lfcc_best.pth")
history_path = os.path.join(save_dir, "light_cnn_lfcc_history.json")

best_eer = float("inf")
best_auc = 0.0
best_thresh = 0.5
training_history = []

print(f"Starting Training: {epochs} Epochs | Device: {device} | AMP: {use_amp}")
print("-" * 75)

for epoch in range(1, epochs + 1):
    start_time = time.time()
    model.train()
    running_loss = 0.0

    for x_batch, y_batch in train_loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(device_type=device.type, enabled=use_amp):
            logits = model(x_batch)
            loss = criterion(logits, y_batch)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * x_batch.size(0)

    epoch_train_loss = running_loss / len(train_dataset)
    scheduler.step(epoch)

    val_eer, val_thresh, val_auc, _, _ = evaluate_network(model, dev_loader, device, use_amp)
    elapsed = time.time() - start_time

    record = {
        "epoch": epoch,
        "train_loss": round(epoch_train_loss, 5),
        "val_eer": round(val_eer, 5),
        "val_auc": round(val_auc, 5),
        "val_thresh": round(val_thresh, 5),
        "time_sec": round(elapsed, 1)
    }
    training_history.append(record)

    is_best = val_eer < best_eer
    if is_best:
        best_eer = val_eer
        best_auc = val_auc
        best_thresh = val_thresh
        torch.save({
            "epoch": epoch,
            "state_dict": model.state_dict(),
            "eer": best_eer,
            "auc": best_auc,
            "threshold": best_thresh,
            "model_architecture": "LightCNN-LFCC"
        }, best_checkpoint_path)

    best_indicator = " [BEST SAVED]" if is_best else ""
    print(f"Epoch [{epoch:02d}/{epochs}] | Loss: {epoch_train_loss:.4f} | Dev EER: {val_eer * 100:.2f}% | Dev AUC: {val_auc:.4f} | Time: {elapsed:.0f}s{best_indicator}")

with open(history_path, "w", encoding="utf-8") as f:
    json.dump(training_history, f, indent=2)

print("-" * 75)
print(f"Training Complete. Optimal Dev EER: {best_eer * 100:.2f}% | Dev AUC: {best_auc:.4f}")
print(f"Best Weights Preserved At: {best_checkpoint_path}")


### 9. Scientific Validation and Training Curves
Visualizes empirical training loss, Equal Error Rate progression, and receiver operating characteristic (ROC) curves.

In [ ]:
ep_list = [h["epoch"] for h in training_history]
losses = [h["train_loss"] for h in training_history]
eers = [h["val_eer"] * 100 for h in training_history]
aucs = [h["val_auc"] for h in training_history]

fig, ax = plt.subplots(1, 3, figsize=(18, 5))

ax[0].plot(ep_list, losses, color="steelblue", lw=2, marker="o", ms=4)
ax[0].set_title("Training Loss (Focal)")
ax[0].set_xlabel("Epoch")
ax[0].set_ylabel("Loss")
ax[0].grid(True, alpha=0.3)

ax[1].plot(ep_list, eers, color="firebrick", lw=2, marker="s", ms=4)
ax[1].axhline(min(eers), color="gray", linestyle="--", label=f"Lowest EER: {min(eers):.2f}%")
ax[1].set_title("Development Equal Error Rate (%)")
ax[1].set_xlabel("Epoch")
ax[1].set_ylabel("EER (%)")
ax[1].legend()
ax[1].grid(True, alpha=0.3)

ax[2].plot(ep_list, aucs, color="forestgreen", lw=2, marker="^", ms=4)
ax[2].axhline(max(aucs), color="gray", linestyle="--", label=f"Peak AUC: {max(aucs):.4f}")
ax[2].set_title("Development ROC-AUC")
ax[2].set_xlabel("Epoch")
ax[2].set_ylabel("AUC")
ax[2].legend()
ax[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(save_dir, "training_progression_curves.png"), dpi=200)
plt.show()

checkpoint = torch.load(best_checkpoint_path, map_location=device)
model.load_state_dict(checkpoint["state_dict"])
_, optimal_thresh, _, final_scores, final_targets = evaluate_network(model, dev_loader, device, use_amp)

fpr, tpr, _ = roc_curve(final_targets, final_scores, pos_label=1)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color="darkviolet", lw=2, label=f"ROC Curve (AUC = {checkpoint['auc']:.4f})")
plt.plot([0, 1], [0, 1], color="gray", linestyle=":")
plt.scatter([checkpoint['eer']], [1 - checkpoint['eer']], color="crimson", zorder=5, label=f"EER Operating Point ({checkpoint['eer']*100:.2f}%)")
plt.xlabel("False Positive Rate (FPR)")
plt.ylabel("True Positive Rate (TPR)")
plt.title("ROC Curve on ASVspoof 2019 Development Partition")
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.savefig(os.path.join(save_dir, "roc_curve_dev.png"), dpi=200)
plt.show()


### 10. Granular Attack Taxonomy Vulnerability Analysis
Evaluates countermeasure robustness across individual spoofing generation algorithms (A01 through A06).

In [ ]:
dev_eval_df = dev_df.copy()
dev_eval_df["spoof_score"] = final_scores
dev_eval_df["predicted_label"] = (final_scores >= optimal_thresh).astype(int)

attack_mapping = {
    "A01": "TTS: Neural Acoustic (AR RNN) + WaveNet",
    "A02": "TTS: Neural Acoustic (AR RNN) + WORLD",
    "A03": "TTS: Concatenative Unit Selection",
    "A04": "VC: Formant / Pitch Shifting + STRAIGHT",
    "A05": "VC: Variational Autoencoder (VAE)",
    "A06": "VC: Transfer Function Regression + WORLD"
}

breakdown = []
for atk_id in ["A01", "A02", "A03", "A04", "A05", "A06"]:
    sub = dev_eval_df[dev_eval_df["attack_id"] == atk_id]
    if len(sub) == 0:
        continue
    correct = (sub["predicted_label"] == 1).sum()
    acc = correct / len(sub)
    mean_prob = sub["spoof_score"].mean()
    breakdown.append({
        "Attack ID": atk_id,
        "Algorithm Family": attack_mapping.get(atk_id, "Unknown"),
        "Total Utterances": len(sub),
        "Detection Accuracy (%)": f"{acc * 100:.2f}%",
        "Mean Spoof Probability": f"{mean_prob:.4f}"
    })

bonafide_sub = dev_eval_df[dev_eval_df["key"] == "bonafide"]
bon_correct = (bonafide_sub["predicted_label"] == 0).sum()
bon_acc = bon_correct / len(bonafide_sub)
breakdown.append({
    "Attack ID": "Bonafide",
    "Algorithm Family": "Authentic Human Speech (VCTK)",
    "Total Utterances": len(bonafide_sub),
    "Detection Accuracy (%)": f"{bon_acc * 100:.2f}%",
    "Mean Spoof Probability": f"{bonafide_sub['spoof_score'].mean():.4f}"
})

print(pd.DataFrame(breakdown).to_string(index=False))


### 11. End-to-End Inference Verification on Real Test Utterances
Demonstrates forensic scoring on individual test audio files.

In [ ]:
def predict_audio_file(file_path, model, device, threshold):
    model.eval()
    raw = load_raw_audio(file_path)
    proc = preprocess_audio(raw, is_train=False)
    feat = extract_lfcc(proc)
    tensor = torch.from_numpy(feat).unsqueeze(0).unsqueeze(0).to(device)
    with torch.no_grad():
        with torch.amp.autocast(device_type=device.type, enabled=use_amp):
            prob = torch.softmax(model(tensor), dim=1)[0, 1].item()
    decision = "SPOOF (SYNTHETIC)" if prob >= threshold else "BONAFIDE (AUTHENTIC)"
    confidence = prob if prob >= threshold else 1.0 - prob
    return {
        "file_name": os.path.basename(file_path),
        "decision": decision,
        "spoof_probability": f"{prob:.4f}",
        "confidence": f"{confidence * 100:.2f}%"
    }

bonafide_sample = dev_df[dev_df["key"] == "bonafide"].iloc[0]["file_path"]
spoof_sample = dev_df[dev_df["key"] == "spoof"].iloc[0]["file_path"]

print("Case 1: Ground Truth Authentic Speech")
print(predict_audio_file(bonafide_sample, model, device, optimal_thresh))
print("\nCase 2: Ground Truth Deepfake Speech")
print(predict_audio_file(spoof_sample, model, device, optimal_thresh))


### 12. Artifacts Summary and Export Verification
Verifies that model checkpoints, historical logs, and diagnostic plots are saved to `/kaggle/working`.

In [ ]:
print("Generated Artifacts in /kaggle/working:")
print("-" * 50)
for entry in os.listdir(save_dir):
    full_path = os.path.join(save_dir, entry)
    if os.path.isfile(full_path):
        size_kb = os.path.getsize(full_path) / 1024
        print(f"  {entry:<35} | {size_kb:>8.1f} KB")
